#чистовик

In [ ]:
# from google.colab import drive
# from pathlib import Path
# drive.mount("/content/drive")

Mounted at /content/drive


In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import re
import glob
import json
import yaml
import xml.etree.ElementTree as ET
import multiprocessing as mp
from pathlib import Path
from itertools import combinations
from collections import Counter
from concurrent.futures import ProcessPoolExecutor

import pandas as pd
import numpy as np
import openpyxl
import matplotlib.pyplot as plt

# DATA_DIR = Path("/content/drive/MyDrive/Airlines")
# CACHE_PATH = Path("/content/boarding_passes_cache.pkl")

DATA_DIR = Path(".")
CACHE_PATH = DATA_DIR / "boarding_passes_cache.pkl"

print("DATA_DIR:", DATA_DIR)
print("Существует:", DATA_DIR.exists())

for p in DATA_DIR.iterdir():
    print(p.name)

TRANSLIT_DICT = {
    'А': 'A', 'Б': 'B', 'В': 'V', 'Г': 'G', 'Д': 'D', 'Е': 'E', 'Ё': 'E',
    'Ж': 'ZH', 'З': 'Z', 'И': 'I', 'Й': 'I', 'К': 'K', 'Л': 'L', 'М': 'M',
    'Н': 'N', 'О': 'O', 'П': 'P', 'Р': 'R', 'С': 'S', 'Т': 'T', 'У': 'U',
    'Ф': 'F', 'Х': 'KH', 'Ц': 'TS', 'Ч': 'CH', 'Ш': 'SH', 'Щ': 'SHCH',
    'Ъ': '', 'Ы': 'Y', 'Ь': '', 'Э': 'E', 'Ю': 'IU', 'Я': 'IA'
}


def transliterate_ru(text: str) -> str:
    if not isinstance(text, str):
        return ""
    return "".join(TRANSLIT_DICT.get(ch, ch) for ch in text.upper())


def normalize_token(name: str) -> str:
    if not isinstance(name, str) or name in ("", "nan", "None"):
        return ""
    s = name.strip().upper()
    s = s.replace("X", "KS")
    s = s.replace("YO", "IO").replace("YA", "IA").replace("YU", "IU").replace("YE", "E")
    s = s.replace("IY", "II").replace("YY", "II").replace("Y", "I")
    s = s.replace("KH", "H").replace("TS", "C").replace("TC", "C")
    return re.sub(r"[^A-Z]", "", s)


class UnionFind:
    def __init__(self):
        self.parent = {}

    def find(self, x):
        if x not in self.parent:
            self.parent[x] = x
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb:
            if ra < rb:
                self.parent[rb] = ra
            else:
                self.parent[ra] = rb








# парсеры




def parse_boarding_data(filepath: Path) -> pd.DataFrame:
    df = pd.read_csv(filepath, sep=";", na_values=["Not presented", "N/A", "", "None"])
    df["birth_date"] = pd.to_datetime(df["PassengerBirthDate"], format="%m/%d/%Y", errors="coerce").dt.strftime("%Y-%m-%d")
    df["flight_date"] = pd.to_datetime(df["FlightDate"], format="%Y-%m-%d", errors="coerce").dt.strftime("%Y-%m-%d")
    df["first_name"] = df["PassengerFirstName"].str.strip().str.upper()
    df["second_name"] = df["PassengerSecondName"].str.strip().str.upper()
    df["last_name"] = df["PassengerLastName"].str.strip().str.upper()
    df["sex"] = df["PassengerSex"].map({"Male": "M", "Female": "F"})
    df["doc_clean"] = df["PassengerDocument"].astype(str).str.replace(r"\s+", "", regex=True).replace("nan", np.nan)
    df["flight_number"] = df["FlightNumber"].str.strip().str.upper()
    df["ticket_number"] = df["TicketNumber"].astype(str).str.strip().str.replace(r"\.0$", "", regex=True).replace("nan", np.nan)
    df["booking_code"] = df["BookingCode"].str.strip().str.upper()
    df["source"] = "BoardingData_CSV"

    cols = [
        "source", "first_name", "second_name", "last_name", "sex", "birth_date",
        "PassengerDocument", "doc_clean", "booking_code", "ticket_number",
        "flight_date", "FlightTime", "flight_number", "Destination", "Baggage", "CodeShare"
    ]
    return df[cols].rename(columns={
        "PassengerDocument": "doc_raw",
        "FlightTime": "depart_time",
        "Destination": "dest_city",
        "Baggage": "baggage",
        "CodeShare": "codeshare"
    })


def parse_sirena_tab(filepath: Path) -> pd.DataFrame:
    colspecs = [
        (0, 60), (60, 72), (72, 84), (84, 96), (96, 108), (108, 120),
        (120, 126), (126, 132), (132, 138), (138, 144), (144, 150),
        (150, 168), (168, 180), (180, 186), (186, 192), (192, 198),
        (198, 204), (204, 216), (216, 276), (276, 336)
    ]
    names = [
        "pax_name_ru", "birth_date", "flight_date", "depart_time",
        "arrival_date", "arrival_time", "flight_number", "codeshare",
        "dep_airport", "arr_airport", "booking_code", "ticket_number",
        "doc_raw", "seat", "meal", "travel_class", "fare", "baggage",
        "additional_info", "agent_info"
    ]
    df = pd.read_fwf(filepath, colspecs=colspecs, names=names, skiprows=1, na_values=["N/A", "None", ""])
    fio_split = df["pax_name_ru"].str.strip().str.split(r"\s+", expand=True)
    df["last_name"] = fio_split[0].apply(transliterate_ru)
    df["first_name"] = fio_split[1].apply(transliterate_ru)
    df["second_name"] = fio_split[2].apply(transliterate_ru) if 2 in fio_split.columns else np.nan
    df["ff_card"] = df["additional_info"].astype(str).str.extract(r"FF#([A-Z0-9]{2}\s+\d+)", expand=False)
    df["doc_clean"] = df["doc_raw"].astype(str).str.replace(r"\s+", "", regex=True).replace("nan", np.nan)
    df["ticket_number"] = df["ticket_number"].astype(str).str.replace(r"\.0$", "", regex=True).replace("nan", np.nan)
    df["source"] = "Sirena_TAB"
    return df


def parse_pointz_xml(filepath: Path) -> pd.DataFrame:
    tree = ET.parse(filepath)
    root = tree.getroot()
    rows = []
    for user in root.findall("user"):
        uid = user.attrib.get("uid")
        name_el = user.find("name")
        first_name = name_el.attrib.get("first", "").strip().upper() if name_el is not None else None
        last_name = name_el.attrib.get("last", "").strip().upper() if name_el is not None else None
        cards_el = user.find("cards")
        if cards_el is None:
            continue
        for card in cards_el.findall("card"):
            card_number = re.sub(r"\s+", " ", card.attrib.get("number", "").strip().upper())
            bonus_prog = card.findtext("bonusprogramm", default="").strip()
            activities_el = card.find("activities")
            if activities_el is None:
                continue
            for act in activities_el.findall("activity"):
                rows.append({
                    "source": "Pointz_XML",
                    "uid": uid,
                    "first_name": first_name,
                    "last_name": last_name,
                    "ff_card": card_number,
                    "bonus_program": bonus_prog,
                    "flight_number": act.findtext("Code", default="").strip().upper(),
                    "flight_date": act.findtext("Date", default="").strip(),
                    "dep_airport": act.findtext("Departure", default="").strip().upper(),
                    "arr_airport": act.findtext("Arrival", default="").strip().upper(),
                    "fare": act.findtext("Fare", default="").strip().upper()
                })
    return pd.DataFrame(rows)


def parse_skyteam_yaml(filepath: Path) -> pd.DataFrame:
    Loader = getattr(yaml, "CSafeLoader", yaml.SafeLoader)
    with open(filepath, "r", encoding="utf-8") as f:
        data = yaml.load(f, Loader=Loader)
    rows = []
    for flight_date, flights in data.items():
        if not isinstance(flights, dict):
            continue
        for flight_num, flight_info in flights.items():
            dep_airport = flight_info.get("FROM")
            arr_airport = flight_info.get("TO")
            status = flight_info.get("STATUS")
            ff_dict = flight_info.get("FF", {})
            if isinstance(ff_dict, dict):
                for ff_card, ff_details in ff_dict.items():
                    rows.append({
                        "source": "SkyTeam_YAML",
                        "flight_date": str(flight_date).strip(),
                        "flight_number": str(flight_num).strip().upper(),
                        "dep_airport": str(dep_airport).strip().upper() if dep_airport else np.nan,
                        "arr_airport": str(arr_airport).strip().upper() if arr_airport else np.nan,
                        "flight_status": str(status).strip().upper() if status else np.nan,
                        "ff_card": re.sub(r"\s+", " ", str(ff_card).strip().upper()),
                        "travel_class": ff_details.get("CLASS"),
                        "fare": ff_details.get("FARE")
                    })
    return pd.DataFrame(rows)


def parse_forum_json(filepath: Path):
    with open(filepath, "r", encoding="utf-8") as f:
        data = json.load(f)
    flight_rows, identity_rows = [], []
    for prof in data.get("Forum Profiles", []):
        nickname = prof.get("NickName")
        sex_raw = prof.get("Sex")
        sex = "M" if sex_raw == "Male" else ("F" if sex_raw == "Female" else np.nan)
        real_name = prof.get("Real Name") or {}
        first_name = (real_name.get("First Name") or "").strip().upper() or np.nan
        last_name = (real_name.get("Last Name") or "").strip().upper() or np.nan

        passports = []
        for doc_obj in (prof.get("Travel Documents") or []):
            if isinstance(doc_obj, dict) and doc_obj.get("Passports"):
                raw_p = doc_obj["Passports"]
                if isinstance(raw_p, list):
                    passports.extend([re.sub(r"\s+", "", str(p)) for p in raw_p if p])
                else:
                    passports.append(re.sub(r"\s+", "", str(raw_p)))

        cards = []
        for lp in (prof.get("Loyality Programm") or []):
            if isinstance(lp, dict):
                prog = (lp.get("programm") or "").strip().upper()
                num = (lp.get("Number") or "").strip()
                if prog and num:
                    cards.append(f"{prog} {num}")

        for card in (cards or [np.nan]):
            for doc in (passports or [np.nan]):
                identity_rows.append({
                    "forum_uid": nickname, "first_name": first_name, "last_name": last_name,
                    "sex": sex, "ff_card": card, "doc_clean": doc
                })

        first_card = cards[0] if len(cards) == 1 else np.nan
        first_doc = passports[0] if len(passports) == 1 else np.nan
        for fl in (prof.get("Registered Flights") or []):
            dep = fl.get("Departure") or {}
            arr = fl.get("Arrival") or {}
            flight_rows.append({
                "source": "Forum_JSON", "forum_uid": nickname,
                "first_name": first_name, "last_name": last_name, "sex": sex,
                "ff_card": first_card, "doc_clean": first_doc,
                "flight_date": (fl.get("Date") or "").strip(),
                "flight_number": (fl.get("Flight") or "").strip().upper(),
                "dep_airport": (dep.get("Airport") or "").strip().upper(),
                "dep_city": (dep.get("City") or "").strip(),
                "arr_airport": (arr.get("Airport") or "").strip().upper(),
                "dest_city": (arr.get("City") or "").strip(),
                "codeshare": str(fl.get("Codeshare"))
            })
    return pd.DataFrame(flight_rows), pd.DataFrame(identity_rows)


def _clean_val(v):
    if v is None:
        return np.nan
    s = str(v).strip()
    return np.nan if s in ("", "N/A", "None", "Not presented") else s


def _parse_single_xlsx(filepath: str) -> list:
    wb = openpyxl.load_workbook(filepath, read_only=True, data_only=True)
    records = []
    for sheetname in wb.sheetnames:
        ws = wb[sheetname]
        grid = list(ws.iter_rows(min_row=1, max_row=13, min_col=1, max_col=8, values_only=True))
        if len(grid) < 13:
            continue
        seq_num = _clean_val(grid[0][7])
        title = _clean_val(grid[2][0])
        full_name = _clean_val(grid[2][1])
        travel_class = _clean_val(grid[2][7])
        flight_num = _clean_val(grid[4][0])
        dep_city = _clean_val(grid[4][3])
        arr_city = _clean_val(grid[4][7])
        dep_airport = _clean_val(grid[6][3])
        arr_airport = _clean_val(grid[6][7])
        flight_date = _clean_val(grid[8][0])
        depart_time = _clean_val(grid[8][2])
        seat = _clean_val(grid[10][7])
        pnr = _clean_val(grid[12][1])
        eticket = _clean_val(grid[12][4])
        ff_card = _clean_val(grid[12][7])

        first_name, last_name = np.nan, np.nan
        if isinstance(full_name, str):
            parts = full_name.upper().split()
            if len(parts) >= 2:
                first_name, last_name = parts[0], parts[-1]
            elif len(parts) == 1:
                last_name = parts[0]

        sex = "M" if title == "MR" else ("F" if title in ("MRS", "MS", "MISS") else np.nan)
        if isinstance(eticket, str) and eticket.endswith(".0"):
            eticket = eticket[:-2]

        records.append({
            "source": "BoardingPass_XLSX", "first_name": first_name, "last_name": last_name,
            "sex": sex, "flight_date": flight_date, "depart_time": depart_time,
            "flight_number": flight_num.upper() if isinstance(flight_num, str) else np.nan,
            "dep_airport": dep_airport.upper() if isinstance(dep_airport, str) else np.nan,
            "arr_airport": arr_airport.upper() if isinstance(arr_airport, str) else np.nan,
            "dep_city": dep_city, "dest_city": arr_city, "travel_class": travel_class,
            "booking_code": pnr, "ticket_number": eticket, "seat": seat,
            "sequence": seq_num, "ff_card": ff_card
        })
    wb.close()
    return records


def load_or_parse_xlsx(data_dir: Path) -> pd.DataFrame:
    if CACHE_PATH.exists():
        return pd.read_pickle(CACHE_PATH)
    xlsx_files = sorted(glob.glob("/content/YourBoardingPassDotAero/**/*.xlsx", recursive=True))
    all_records = []
    ctx = mp.get_context("fork")
    with ProcessPoolExecutor(max_workers=os.cpu_count(), mp_context=ctx) as executor:
        for file_records in executor.map(_parse_single_xlsx, xlsx_files, chunksize=4):
            all_records.extend(file_records)
    df_xlsx = pd.DataFrame(all_records)
    df_xlsx.to_pickle(CACHE_PATH)
    return df_xlsx







def main():
    print("Step 1: Parsing all 6 data sources...")
    df_boarding = parse_boarding_data(DATA_DIR / "BoardingData.csv")
    df_sirena = parse_sirena_tab(DATA_DIR / "Sirena-export-fixed.tab")
    df_xml = parse_pointz_xml(DATA_DIR / "PointzAggregator-AirlinesData.xml")
    df_yaml = parse_skyteam_yaml(DATA_DIR / "SkyTeam-Exchange.yaml")
    df_forum_flights, df_forum_id = parse_forum_json(DATA_DIR / "FrequentFlyerForum-Profiles.json")
    df_xlsx = load_or_parse_xlsx(DATA_DIR)

    common_cols = [
        "source", "first_name", "second_name", "last_name", "sex", "birth_date",
        "doc_clean", "ff_card", "ticket_number", "booking_code",
        "flight_date", "depart_time", "flight_number", "dep_airport", "arr_airport",
        "dep_city", "dest_city", "travel_class", "fare", "baggage", "codeshare",
        "forum_uid", "uid"
    ]

    def align_df(df: pd.DataFrame) -> pd.DataFrame:
        res = pd.DataFrame(index=df.index)
        for c in common_cols:
            res[c] = df[c] if c in df.columns else np.nan
        return res

    df_all = pd.concat([
        align_df(df_boarding), align_df(df_sirena),
        align_df(df_xml.dropna(subset=["flight_number"])),
        align_df(df_yaml), align_df(df_forum_flights), align_df(df_xlsx)
    ], ignore_index=True)

    print(f"Total raw records loaded: {len(df_all):,}")

    #  Cleaning & Normalization
    for col in ["doc_clean", "ff_card", "ticket_number", "booking_code", "forum_uid", "uid", "birth_date"]:
        df_all[col] = df_all[col].astype(str).str.strip().replace({"nan": np.nan, "None": np.nan, "": np.nan, "N/A": np.nan})

    # Fix REA -> ICN airport code bug
    df_all["dep_airport"] = df_all["dep_airport"].replace({"REA": "ICN"})
    df_all["arr_airport"] = df_all["arr_airport"].replace({"REA": "ICN"})


    # Проверяем, встречается ли один номер рейса на разных маршрутах.
    route_check = (
        df_all
        .dropna(subset=["flight_number", "dep_airport", "arr_airport"])
        .assign(
            route=lambda x:
                x["dep_airport"].astype(str)
                + " -> "
                + x["arr_airport"].astype(str)
        )
        .groupby("flight_number")["route"]
        .nunique()
        .sort_values(ascending=False)
    )

    ambiguous_flights = route_check[route_check > 1]

    print(
        "Номеров рейсов с несколькими маршрутами:",
        len(ambiguous_flights)
    )

    display(ambiguous_flights.head(20))

    # Enrich missing airports & departure times via flight reference table
    flight_ref = (
        df_all.dropna(subset=["flight_number"])
        .groupby("flight_number")
        .agg(
            ref_dep=("dep_airport", lambda s: s.dropna().mode().iloc[0] if not s.dropna().empty else np.nan),
            ref_arr=("arr_airport", lambda s: s.dropna().mode().iloc[0] if not s.dropna().empty else np.nan),
            ref_time=("depart_time", lambda s: s.dropna().mode().iloc[0] if not s.dropna().empty else np.nan)
        )
    )
    df_all["dep_airport"] = df_all["dep_airport"].fillna(df_all["flight_number"].map(flight_ref["ref_dep"]))
    df_all["arr_airport"] = df_all["arr_airport"].fillna(df_all["flight_number"].map(flight_ref["ref_arr"]))
    df_all["depart_time"] = df_all["depart_time"].fillna(df_all["flight_number"].map(flight_ref["ref_time"]))

    df_all["fn_norm"] = df_all["first_name"].apply(normalize_token)
    df_all["ln_norm"] = df_all["last_name"].apply(normalize_token)
    df_all["sn_initial"] = df_all["second_name"].apply(lambda x: normalize_token(x)[:1] if pd.notna(x) else "")

    valid_both = (df_all["fn_norm"].str.len() > 1) & (df_all["ln_norm"].str.len() > 1)
    df_all["full_name_norm"] = np.where(
        valid_both,
        np.minimum(df_all["fn_norm"], df_all["ln_norm"]) + "_" + np.maximum(df_all["fn_norm"], df_all["ln_norm"]),
        np.nan
    )

    ###
    doc_stats = (
        df_all.dropna(subset=["doc_clean"])
        .groupby("doc_clean")
        .agg(
            n_names=("full_name_norm", "nunique"),
            n_birth_dates=("birth_date", "nunique"),
        )
    )

    compromised_docs = set(
        doc_stats[
            (doc_stats["n_names"] > 1)
            | (doc_stats["n_birth_dates"] > 1)
        ].index
    )

    ff_stats = (
        df_all.dropna(subset=["ff_card"])
        .groupby("ff_card")
        .agg(
            n_names=("full_name_norm", "nunique"),
            n_birth_dates=("birth_date", "nunique"),
        )
    )

    compromised_cards = set(
        ff_stats[
            (ff_stats["n_names"] > 1)
            | (ff_stats["n_birth_dates"] > 1)
        ].index
    )
###

    df_all["doc_len"] = df_all["doc_clean"].str.len()
    df_all["doc_dom"] = np.where(df_all["doc_len"] == 10, df_all["doc_clean"], np.nan)
    df_all["doc_intl"] = np.where(df_all["doc_len"] == 9, df_all["doc_clean"], np.nan)

    #  Entity Resolution & Graph Linking
    print("Step 3: Running Entity Resolution (Union-Find + Trajectory Matching)...")

    uf = UnionFind()
    for row in df_forum_id.itertuples():
        nodes = [f"FUID:{row.forum_uid}"]
        if pd.notna(row.ff_card) and row.ff_card not in compromised_cards:
            nodes.append(f"FF:{row.ff_card}")
        if pd.notna(row.doc_clean) and row.doc_clean not in compromised_docs:
            nodes.append(f"DOC:{row.doc_clean}")
        for i in range(1, len(nodes)):
            uf.union(nodes[0], nodes[i])

    ticket_links = df_all.dropna(subset=["ticket_number"])[["ticket_number", "doc_clean", "ff_card"]].drop_duplicates()
    for row in ticket_links.itertuples():
        t_node = f"TKT:{row.ticket_number}"
        if pd.notna(row.doc_clean) and row.doc_clean not in compromised_docs:
            uf.union(t_node, f"DOC:{row.doc_clean}")
        if pd.notna(row.ff_card) and row.ff_card not in compromised_cards:
            uf.union(t_node, f"FF:{row.ff_card}")

    id_subset = df_all[["doc_clean", "ff_card", "uid", "forum_uid"]].dropna(how="all").drop_duplicates()
    for row in id_subset.itertuples():
        nodes = []
        if pd.notna(row.doc_clean) and row.doc_clean not in compromised_docs:
            nodes.append(f"DOC:{row.doc_clean}")
        if pd.notna(row.ff_card) and row.ff_card not in compromised_cards:
            nodes.append(f"FF:{row.ff_card}")
        if pd.notna(row.uid):
            nodes.append(f"XUID:{row.uid}")
        if pd.notna(row.forum_uid):
            nodes.append(f"FUID:{row.forum_uid}")
        for i in range(1, len(nodes)):
            uf.union(nodes[0], nodes[i])

    row_nodes = pd.Series(index=df_all.index, dtype=object)
    for prefix, col, bad_set in [
        ("DOC:", "doc_clean", compromised_docs),
        ("TKT:", "ticket_number", set()),
        ("FF:", "ff_card", compromised_cards),
        ("XUID:", "uid", set()),
        ("FUID:", "forum_uid", set())
    ]:
        mask = row_nodes.isna() & df_all[col].notna() & (~df_all[col].isin(bad_set))
        row_nodes.loc[mask] = prefix + df_all.loc[mask, col]

    mask_empty = row_nodes.isna()
    row_nodes.loc[mask_empty] = [f"ROW:{i}" for i in df_all.index[mask_empty]]
    df_all["row_node"] = row_nodes

    # Link records within same flight if no namesake collision on board
    flight_name_stats = (
        df_all.dropna(subset=["flight_date", "flight_number", "full_name_norm"])
        .groupby(["flight_date", "flight_number", "full_name_norm"])
        .agg(
            n_dom=("doc_dom", "nunique"), n_intl=("doc_intl", "nunique"),
            n_births=("birth_date", "nunique"),
            n_sn=("sn_initial", lambda s: len(set(x for x in s if x != ""))),
            n_tkts=("ticket_number", "nunique")
        )
    )
    safe_flight_names = flight_name_stats[
        (flight_name_stats["n_dom"] <= 1) & (flight_name_stats["n_intl"] <= 1) &
        (flight_name_stats["n_births"] <= 1) & (flight_name_stats["n_sn"] <= 1) &
        (flight_name_stats["n_tkts"] <= 1)
    ].index

    df_safe_flights = df_all.dropna(subset=["flight_date", "flight_number", "full_name_norm"]).set_index(
        ["flight_date", "flight_number", "full_name_norm"]
    )
    df_safe_flights = df_safe_flights.loc[df_safe_flights.index.intersection(safe_flight_names)].reset_index()
    for _, grp in df_safe_flights[["flight_date", "flight_number", "full_name_norm", "row_node"]].drop_duplicates().groupby(
        ["flight_date", "flight_number", "full_name_norm"], sort=False
    ):
        nodes = grp["row_node"].values
        for n in nodes[1:]:
            uf.union(nodes[0], n)

    # Link by (full_name_norm + birth_date)
    name_birth_groups = (
        df_all.dropna(subset=["full_name_norm", "birth_date"])
        .groupby(["full_name_norm", "birth_date"])
        .agg(
            n_dom=("doc_dom", "nunique"), n_intl=("doc_intl", "nunique"),
            n_sn=("sn_initial", lambda s: len(set(x for x in s if x != "")))
        )
    )
    safe_nb = name_birth_groups[
        (name_birth_groups["n_dom"] <= 1) & (name_birth_groups["n_intl"] <= 1) & (name_birth_groups["n_sn"] <= 1)
    ].index
    df_nb = df_all.dropna(subset=["full_name_norm", "birth_date"]).set_index(["full_name_norm", "birth_date"])
    df_nb = df_nb.loc[df_nb.index.intersection(safe_nb)].reset_index()
    for _, grp in df_nb[["full_name_norm", "birth_date", "row_node"]].drop_duplicates().groupby(
        ["full_name_norm", "birth_date"], sort=False
    ):
        nodes = grp["row_node"].values
        for n in nodes[1:]:
            uf.union(nodes[0], n)

    df_all["pid_clean"] = df_all["row_node"].map(uf.find)

    # Trajectory deanonymization (>=5 shared flights)
    flights_tmp = (
        df_all.dropna(subset=["flight_date", "flight_number"])
        [["pid_clean", "flight_date", "flight_number"]]
        .drop_duplicates()
    )
    pax_meta = df_all.groupby("pid_clean").agg(
        name=("full_name_norm", lambda s: s.dropna().iloc[0] if not s.dropna().empty else np.nan),
        total_f=("flight_number", "nunique")
    )
    active_pids = set(pax_meta[pax_meta["total_f"] >= 4].index)
    rosters_clean = (
        flights_tmp[flights_tmp["pid_clean"].isin(active_pids)]
        .groupby(["flight_date", "flight_number"])["pid_clean"]
        .apply(list)
    )
    pair_cnt = Counter()
    for plist in rosters_clean:
        if 2 <= len(plist) <= 25:
            for p1, p2 in combinations(sorted(set(plist)), 2):
                pair_cnt[(p1, p2)] += 1

    for (p1, p2), c in pair_cnt.items():
        if c >= 5 and (pd.isna(pax_meta.loc[p1, "name"]) ^ pd.isna(pax_meta.loc[p2, "name"])):
            uf.union(p1, p2)

    df_all["person_id_final"] = df_all["row_node"].map(uf.find)

    # Deduplicate flights & propagate attributes
    df_clean_flights = (
        df_all.dropna(subset=["flight_date", "flight_number"])
        .sort_values(["person_id_final", "flight_date", "depart_time"])
        .groupby(["person_id_final", "flight_date", "flight_number"], as_index=False)
        .first()
    )

    attr_cols = ["full_name_norm", "first_name", "second_name", "last_name", "sex", "birth_date", "doc_clean", "ff_card"]
    person_attrs = df_clean_flights.groupby("person_id_final")[attr_cols].first()
    for col in attr_cols:
        df_clean_flights[col] = df_clean_flights[col].fillna(df_clean_flights["person_id_final"].map(person_attrs[col]))


    # Step 4: Export clean flight-level data for the Data Analyst
    # На этом граница ответственности Data Engineer:
    # одна строка = один уникальный перелёт одного person_id_final.
    export_flight_cols = [
        "person_id_final", "full_name_norm", "first_name", "second_name", "last_name",
        "sex", "birth_date", "doc_clean", "ff_card", "ticket_number", "booking_code",
        "flight_date", "depart_time", "flight_number", "dep_airport", "arr_airport",
        "dep_city", "dest_city", "travel_class", "fare", "baggage", "codeshare"
    ]
    existing_cols = [c for c in export_flight_cols if c in df_clean_flights.columns]
    engineer_output = df_clean_flights[existing_cols].copy()

    engineer_output.to_csv(
        DATA_DIR / "yoyoflot_flights_merged_clean.csv",
        index=False
    )
    engineer_output.to_csv(
        DATA_DIR / "yoyoflot_flights_merged_clean.csv.gz",
        index=False,
        compression="gzip"
    )

    print(
        f"Data Engineer finished: exported {len(engineer_output):,} "
        f"unique passenger-flight records."
    )
    print("Output: yoyoflot_flights_merged_clean.csv / .csv.gz")
    return engineer_output


if __name__ == "__main__":
    df_clean_flights = main()


DATA_DIR: .
Существует: True
FrequentFlyerForum-Profiles.json
Yoyoflot_Final_Clean (6).ipynb
Untitled.ipynb
Skyteam_Timetable.pdf
report_summary.png
12-17.ipynb
SkyTeam-Exchange.yaml
PointzAggregator-AirlinesData.xml
21-24.ipynb
boarding_passes_cache.pkl
BoardingData.csv
yoyoflot_flights_merged_clean.csv
app.py
Sirena-export-fixed.tab
25-28.ipynb
29.ipynb
YourBoardingPassDotAero.zip
18-20.ipynb
AllInOne.ipynb
yoyoflot_passengers_summary.csv
cufile.log
.ipynb_checkpoints
YourBoardingPassDotAero
.venv
30.ipynb
yoyoflot_flights_merged_clean.csv.gz
Step 1: Parsing all 6 data sources...
Total raw records loaded: 4,358,805
Номеров рейсов с несколькими маршрутами: 8172


flight_number
DL3336    29
DL678     25
DL436     25
DL3286    24
DL426     24
DL3320    24
DL3293    23
DL3360    23
DL797     23
DL443     23
DL3294    22
DL434     22
DL3289    22
DL3392    22
DL3334    22
DL3296    22
DL3295    22
DL861     22
DL3525    22
DL442     22
Name: route, dtype: int64

Step 3: Running Entity Resolution (Union-Find + Trajectory Matching)...
Data Engineer finished: exported 2,266,112 unique passenger-flight records.
Output: yoyoflot_flights_merged_clean.csv / .csv.gz


In [2]:


from pathlib import Path
import pandas as pd
import numpy as np

DATA_DIR = Path(".")

csv_path = DATA_DIR / "yoyoflot_flights_merged_clean.csv"
gz_path = DATA_DIR / "yoyoflot_flights_merged_clean.csv.gz"
input_path = csv_path if csv_path.exists() else gz_path

df = pd.read_csv(input_path, dtype={
        "doc_clean": str,
        "ticket_number": str,
        "ff_card": str,
        "booking_code": str,
    }, low_memory=False,)

print(f"Получено перелётов: {len(df):,}")
print(f"Уникальных профилей: {df['person_id_final'].nunique():,}")
print("Поля:", list(df.columns))

#подтягиваем данные (заполнение пропусков)
attr_cols = [
    "full_name_norm", "first_name", "second_name", "last_name",
    "sex", "birth_date", "doc_clean", "ff_card"
]
attr_cols = [c for c in attr_cols if c in df.columns]

person_attrs = df.groupby("person_id_final")[attr_cols].first()
for col in attr_cols:
    df[col] = df[col].fillna(df["person_id_final"].map(person_attrs[col]))

# 3. Гипотеза №1: поведенческие аномалии.


#сортируем перелеты по дате для каждого пассажира
df = df.sort_values(["person_id_final", "flight_date", "depart_time"]).reset_index(drop=True)

AIRPORT_TO_CITY_HUB = {
    # Basel / Mulhouse
    "BSL": "EAP",
    "MLH": "EAP",

    # Berlin — для исторических данных 2017 года
    "SXF": "BER",
    "TXL": "BER",

    # Buenos Aires
    "AEP": "BUE",
    "EZE": "BUE",

    # Chicago
    "MDW": "CHI",
    "ORD": "CHI",

    # Dallas
    "DAL": "DFW",

    # Houston
    "IAH": "HOU",

    # Istanbul
    "SAW": "IST",

    # Kyiv
    "KBP": "IEV",

    # London
    "LCY": "LON",
    "LGW": "LON",
    "LHR": "LON",
    "LTN": "LON",
    "SEN": "LON",

    # Milan
    "BGY": "MIL",
    "LIN": "MIL",
    "MXP": "MIL",

    # Moscow
    "DME": "MOW",
    "SVO": "MOW",
    "VKO": "MOW",

    # New York
    "EWR": "NYC",
    "JFK": "NYC",
    "LGA": "NYC",
    "SWF": "NYC",

    # Osaka
    "ITM": "OSA",
    "KIX": "OSA",

    # Paris
    "CDG": "PAR",
    "ORY": "PAR",

    # Shanghai
    "PVG": "SHA",

    # Saint Lucia
    "UVF": "SLU",

    # Oslo
    "TRF": "OSL",

    # Taipei
    "TSA": "TPE",

    # Tenerife
    "TFN": "TCI",
    "TFS": "TCI",

    # Tokyo
    "HND": "TYO",
    "NRT": "TYO",

    # Washington
    "DCA": "WAS",
    "IAD": "WAS",
}

df["dep_hub"] = df["dep_airport"].replace(AIRPORT_TO_CITY_HUB)
df["arr_hub"] = df["arr_airport"].replace(AIRPORT_TO_CITY_HUB)

df["flight_date_dt"] = pd.to_datetime(df["flight_date"], errors="coerce")
df["prev_flight_date_dt"] = df.groupby("person_id_final")["flight_date_dt"].shift(1)
df["prev_arr_hub"] = df.groupby("person_id_final")["arr_hub"].shift(1)
df["days_since_prev"] = (df["flight_date_dt"] - df["prev_flight_date_dt"]).dt.days

# Быстрый разворот:
# следующий известный вылет происходит из того же города, куда пассажир
# только что прилетел, не позднее чем на следующих сутках
df["is_quick_turnaround"] = (df["days_since_prev"].between(0, 1, inclusive="both")
    & (df["dep_hub"] == df["prev_arr_hub"]))

# Разрыв маршрута:
# следующий известный рейс начинается не там, где закончился предыдущий.
# Это индикатор неполноты/разрыва траектории, а не доказательство нарушения.
df["is_teleport"] = (df["dep_hub"].notna() & df["prev_arr_hub"].notna()
    & (df["dep_hub"] != df["prev_arr_hub"]))

baggage = (df["baggage"].astype("string").str.strip().str.upper())

df["no_baggage"] = pd.Series(pd.NA, index=df.index, dtype="boolean")

df.loc[baggage == "0PC", "no_baggage"] = True

df.loc[baggage.isin(["1PC", "2PC", "REGISTERED", "TRANSIT", "DELAYED"]), "no_baggage"] = False

#переделываем базу данных
passengers_summary = (df.groupby("person_id_final").agg(
    full_name=("full_name_norm", "first"),
    sex=("sex", "first"),
    birth_date=("birth_date", "first"),
    primary_doc=("doc_clean", "first"),
    primary_ff_card=("ff_card", "first"),

    n_names=("full_name_norm", "nunique"),
    n_birth_dates=("birth_date", "nunique"),
    n_docs=("doc_clean", "nunique"),
    n_ff_cards=("ff_card", "nunique"),



    total_flights=("flight_number", "count"),
    quick_turnarounds=("is_quick_turnaround", "sum"),
    teleports=("is_teleport", "sum"),
    no_baggage_flights=("no_baggage", "sum"),
    known_baggage_flights=("no_baggage", "count"),
    first_flight=("flight_date", "min"),
    last_flight=("flight_date", "max"),))

passenger_airports = pd.concat([
    df[["person_id_final", "dep_airport"]].rename(columns={"dep_airport": "airport"}),
    df[["person_id_final", "arr_airport"]].rename(columns={"arr_airport": "airport"})])

unique_airports = (passenger_airports.dropna(subset=["airport"]).drop_duplicates()
    .groupby("person_id_final")["airport"].nunique().rename("unique_airports"))

passengers_summary = passengers_summary.join(unique_airports)

passengers_summary["unique_airports"] = (passengers_summary["unique_airports"].fillna(0).astype(int))

known_baggage = passengers_summary["known_baggage_flights"].replace(0, np.nan)

passengers_summary["no_baggage_pct"] = (passengers_summary["no_baggage_flights"]
    / known_baggage* 100).round(1)

passengers_summary["baggage_known_pct"] = (passengers_summary["known_baggage_flights"]
    / passengers_summary["total_flights"].clip(lower=1)* 100).round(1)


#2 гипотеза - конфликты идентичности.
#один документ или одна карта встречается у нескольких личностей
doc_conflicts = (df.dropna(subset=["doc_clean"]).groupby("doc_clean").agg(
        n_names=("full_name_norm", "nunique"),
        n_birth_dates=("birth_date", "nunique"),
        n_profiles=("person_id_final", "nunique"),))

doc_conflicts = doc_conflicts[(doc_conflicts["n_names"] > 1) | (doc_conflicts["n_birth_dates"] > 1)
    | (doc_conflicts["n_profiles"] > 1)].sort_values(["n_profiles", "n_names", "n_birth_dates"], ascending=False)

ff_conflicts = (df.dropna(subset=["ff_card"]).groupby("ff_card").agg(
        n_names=("full_name_norm", "nunique"),
        n_birth_dates=("birth_date", "nunique"),
        n_profiles=("person_id_final", "nunique"),))

ff_conflicts = ff_conflicts[(ff_conflicts["n_names"] > 1) | (ff_conflicts["n_birth_dates"] > 1)
    | (ff_conflicts["n_profiles"] > 1)].sort_values(["n_profiles", "n_names", "n_birth_dates"], ascending=False)

conflict_docs = set(doc_conflicts.index)
conflict_ff = set(ff_conflicts.index)

df["has_doc_conflict"]= df["doc_clean"].isin(conflict_docs)
df["has_ff_conflict"]= df["ff_card"].isin(conflict_ff)

identity_flags = (df.groupby("person_id_final").agg(
    has_doc_conflict=("has_doc_conflict", "max"),
    has_ff_conflict=("has_ff_conflict", "max"),))
passengers_summary = passengers_summary.join(identity_flags)

DOC_CONFLICT_WEIGHT = 100
FF_CONFLICT_WEIGHT = 80

passengers_summary["identity_score"] = (
    passengers_summary["has_doc_conflict"].astype(int) * DOC_CONFLICT_WEIGHT
    + passengers_summary["has_ff_conflict"].astype(int) * FF_CONFLICT_WEIGHT)


#3 гипотеза - «курьерский» поведенческий паттерн

courier_candidates = passengers_summary[
    (passengers_summary["total_flights"] >= 25)
    & (passengers_summary["quick_turnarounds"] >= 4)
    & (passengers_summary["no_baggage_pct"] >= 85)
    & (passengers_summary["baggage_known_pct"] >= 30)
    & (passengers_summary["teleports"] <= 10)].copy()

passengers_summary["no_baggage_signal"] = (passengers_summary["no_baggage_pct"].fillna(0)
    * passengers_summary["baggage_known_pct"]/ 100).round(1)


passengers_summary["courier_score"] = (
    passengers_summary["total_flights"] * 1.2
    + passengers_summary["quick_turnarounds"] * 4.5
    + passengers_summary["no_baggage_signal"] * 0.2).round(1)

courier_candidates = (passengers_summary.loc[courier_candidates.index]
    .sort_values(["courier_score", "quick_turnarounds", "total_flights"],ascending=False,))


#общий рейтинг
passengers_summary["risk_score"] = (
    passengers_summary["identity_score"]
    + passengers_summary["courier_score"]).round(1)

#сохранение
passengers_summary = passengers_summary.sort_values(["risk_score", "quick_turnarounds", "total_flights"], ascending=False,)

passengers_summary.to_csv(DATA_DIR / "yoyoflot_passengers_summary.csv", index=True, index_label="person_id_final",)

# Для дашборда сохраняем также обогащённую таблицу рейсов
# с рассчитанными аналитиком признаками.
export_cols = [
    "person_id_final", "full_name_norm", "first_name", "second_name", "last_name",
    "sex", "birth_date", "doc_clean", "ff_card", "ticket_number", "booking_code",
    "flight_date", "depart_time", "flight_number", "dep_airport", "arr_airport",
    "dep_city", "dest_city", "travel_class", "fare", "baggage", "codeshare",
    "is_quick_turnaround", "is_teleport", "no_baggage",
]
export_cols = [c for c in export_cols if c in df.columns]

df[export_cols].to_csv( DATA_DIR / "yoyoflot_flights_analytics.csv", index=False,)
df[export_cols].to_csv( DATA_DIR / "yoyoflot_flights_analytics.csv.gz", index=False, compression="gzip",)

print("\n=== РЕЗУЛЬТАТ АНАЛИТИКА ===")
print(f"Профилей в summary: {len(passengers_summary):,}")
print(f"Конфликтных документов: {len(doc_conflicts):,}")
print(f"Конфликтных карт лояльности: {len(ff_conflicts):,}")
print(f"Кандидатов по курьерскому фильтру: {len(courier_candidates):,}")

print("\nТоп конфликтных документов:")
display(doc_conflicts.head(10))

print("\nТоп конфликтных карт:")
display(ff_conflicts.head(10))

print("\nТоп кандидатов по поведенческим признакам:")
display(courier_candidates.head(15))


Получено перелётов: 2,266,112
Уникальных профилей: 463,872
Поля: ['person_id_final', 'full_name_norm', 'first_name', 'second_name', 'last_name', 'sex', 'birth_date', 'doc_clean', 'ff_card', 'ticket_number', 'booking_code', 'flight_date', 'depart_time', 'flight_number', 'dep_airport', 'arr_airport', 'dep_city', 'dest_city', 'travel_class', 'fare', 'baggage', 'codeshare']

=== РЕЗУЛЬТАТ АНАЛИТИКА ===
Профилей в summary: 463,872
Конфликтных документов: 1
Конфликтных карт лояльности: 1
Кандидатов по курьерскому фильтру: 0

Топ конфликтных документов:


,n_names,n_birth_dates,n_profiles
doc_clean,,,
8248013778,2,2,2



Топ конфликтных карт:


,n_names,n_birth_dates,n_profiles
ff_card,,,
FB 139888197,2,1,12



Топ кандидатов по поведенческим признакам:


,full_name,sex,birth_date,primary_doc,primary_ff_card,n_names,n_birth_dates,n_docs,n_ff_cards,total_flights,...,first_flight,last_flight,unique_airports,no_baggage_pct,baggage_known_pct,has_doc_conflict,has_ff_conflict,identity_score,no_baggage_signal,courier_score
person_id_final,,,,,,,,,,,,,,,,,,,,,
